# Student AI Tools vs Exam Score Prediction: Model Training

## 1. Project Title

This notebook focuses on training various regression models to predict student exam scores based on their AI tool usage. We will compare model performance, select the best model, and save it for future deployment.

## 2. Import Required Libraries

We import all necessary libraries for data manipulation, machine learning model training, evaluation, and saving.

In [23]:
# Standard libraries for data manipulation and numerical operations
import pandas as pd
import numpy as np

# Libraries for plotting and visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Library for saving and loading Python objects
import joblib

# Scikit-learn modules for machine learning tasks
from sklearn.model_selection import train_test_split # For splitting data into training and testing sets
from sklearn.linear_model import LinearRegression      # Linear Regression model
from sklearn.ensemble import RandomForestRegressor     # Random Forest Regressor model
from sklearn.ensemble import GradientBoostingRegressor # Gradient Boosting Regressor model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score # Evaluation metrics
from sklearn.preprocessing import OneHotEncoder      # For one-hot encoding categorical features
from sklearn.compose import ColumnTransformer        # For applying different transformers to different columns
from sklearn.pipeline import Pipeline                # For creating a pipeline of transformers

## 3. Load the Preprocessed Dataset

We load the preprocessed dataset, which was cleaned and prepared in the previous steps, into a pandas DataFrame. We then inspect its initial rows, shape, and data types to ensure it's ready for modeling.

In [2]:
# Define the path to the dataset
DATASET_PATH = '/content/student_ai_tools_vs_exam_scores.csv'

# Load the dataset
df = pd.read_csv(DATASET_PATH)

# Display the first few rows of the DataFrame
print("First 5 rows of the dataset:")
display(df.head())

# Display the shape of the DataFrame
print(f"\nShape of the dataset: {df.shape}")

# Display the data types of each column
print("\nData types of the columns:")
display(df.info())

First 5 rows of the dataset:


,age,education_level,study_hours_per_day,uses_ai,ai_tools_used,purpose_of_ai,grades_before_ai,grades_after_ai,daily_screen_time_hours
0,19,college,1.4,No,NaN,NaN,62,62,3
1,15,school,3.9,Yes,Copilot,Research,56,61,2
2,15,school,1.9,Yes,Copilot,Homework,75,88,5
3,15,school,2.8,No,NaN,NaN,55,55,3
4,19,college,2.7,No,NaN,NaN,59,59,3



Shape of the dataset: (5000, 9)

Data types of the columns:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   age                      5000 non-null   int64  
 1   education_level          5000 non-null   object 
 2   study_hours_per_day      5000 non-null   float64
 3   uses_ai                  5000 non-null   object 
 4   ai_tools_used            1979 non-null   object 
 5   purpose_of_ai            1979 non-null   object 
 6   grades_before_ai         5000 non-null   int64  
 7   grades_after_ai          5000 non-null   int64  
 8   daily_screen_time_hours  5000 non-null   int64  
dtypes: float64(1), int64(4), object(4)
memory usage: 351.7+ KB


None

## 4. Define Features and Target

We separate the dataset into features (X) and the target variable (y). The target variable `Grades_After_AI` represents the exam scores we want to predict.

In [13]:
# Define features (X) by dropping the target column
X = df.drop(columns=['grades_after_ai'])

# Define the target variable (y)
y = df['grades_after_ai']

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)
print("\nFeatures (X) head:")
display(X.head())
print("\nTarget (y) head:")
display(y.head())

Features (X) shape: (5000, 8)
Target (y) shape: (5000,)

Features (X) head:


,age,education_level,study_hours_per_day,uses_ai,ai_tools_used,purpose_of_ai,grades_before_ai,daily_screen_time_hours
0,19,college,1.4,No,NaN,NaN,62,3
1,15,school,3.9,Yes,Copilot,Research,56,2
2,15,school,1.9,Yes,Copilot,Homework,75,5
3,15,school,2.8,No,NaN,NaN,55,3
4,19,college,2.7,No,NaN,NaN,59,3



Target (y) head:


,grades_after_ai
0,62
1,61
2,88
3,55
4,59


## 5. Train-Test Split

We split the dataset into training and testing sets. This allows us to train our models on one portion of the data and evaluate their performance on unseen data, preventing overfitting.

In [26]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (4000, 8)
X_test shape: (1000, 8)
y_train shape: (4000,)
y_test shape: (1000,)


## 5.5 Preprocessing Categorical Features

Machine learning models cannot directly handle categorical (text-based) features. We need to convert these into numerical representations. Additionally, some categorical columns have missing values that need to be addressed. We will define a custom imputer to handle NaNs based on logic and then use `OneHotEncoder` for conversion.

In [29]:
# Define categorical features that need encoding
categorical_features = ['education_level', 'uses_ai', 'ai_tools_used', 'purpose_of_ai']

# Create a custom imputer function (defined globally for re-use in test section)
def custom_imputer(df_input):
    df_copy = df_input.copy()
    # For 'ai_tools_used' and 'purpose_of_ai'
    for col in ['ai_tools_used', 'purpose_of_ai']:
        # If 'uses_ai' is 'No', implies no AI tools were used, so mark as 'No AI'
        df_copy.loc[df_copy['uses_ai'] == 'No', col] = df_copy.loc[df_copy['uses_ai'] == 'No', col].fillna('No AI')
        # For other cases (uses_ai is 'Yes' or NaN), fill remaining NaNs with 'Unknown'
        df_copy[col] = df_copy[col].fillna('Unknown')
    return df_copy

# Apply custom imputation to both train and test sets before one-hot encoding
X_train_imputed = custom_imputer(X_train)
X_test_imputed = custom_imputer(X_test)

# Create the preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough' # Keep other numerical columns as they are
)

# Fit the preprocessor on the training data and transform both training and testing data
X_train_processed = preprocessor.fit_transform(X_train_imputed)
X_test_processed = preprocessor.transform(X_test_imputed)

# Get feature names after one-hot encoding for better interpretability and for `d7ee6495`
# These names will be used when converting processed arrays back to DataFrames
processed_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features).tolist() + \
                          [col for col in X_train.columns if col not in categorical_features]

# Convert processed data back to DataFrame for models that might prefer it (or for inspection)
X_train_processed = pd.DataFrame(X_train_processed, columns=processed_feature_names, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_processed, columns=processed_feature_names, index=X_test.index)

print("Categorical features preprocessed and One-Hot encoded.")
print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_test_processed shape: {X_test_processed.shape}")
print("\nFirst 5 rows of X_train_processed:")
display(X_train_processed.head())

# Save the preprocessor
MODEL_DIR = 'models/'
os.makedirs(MODEL_DIR, exist_ok=True)
PREPROCESSOR_PATH = os.path.join(MODEL_DIR, 'preprocessor.pkl')
joblib.dump(preprocessor, PREPROCESSOR_PATH)
print(f"Preprocessor saved to {PREPROCESSOR_PATH}")

Categorical features preprocessed and One-Hot encoded.
X_train_processed shape: (4000, 16)
X_test_processed shape: (1000, 16)

First 5 rows of X_train_processed:


,education_level_college,education_level_school,uses_ai_No,uses_ai_Yes,ai_tools_used_ChatGPT,ai_tools_used_Copilot,ai_tools_used_Gemini,ai_tools_used_No AI,purpose_of_ai_Coding,purpose_of_ai_Homework,purpose_of_ai_No AI,purpose_of_ai_Research,age,study_hours_per_day,grades_before_ai,daily_screen_time_hours
4227,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,18.0,4.6,70.0,2.0
4676,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,18.0,3.8,57.0,4.0
800,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,14.0,4.4,70.0,2.0
3671,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,19.0,1.9,60.0,7.0
4193,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,18.0,3.8,57.0,4.0


Preprocessor saved to models/preprocessor.pkl


## 5.5 Preprocessing Categorical Features

Machine learning models, especially linear models, cannot directly handle categorical (text-based) features. We need to convert these into numerical representations. Additionally, some categorical columns have missing values that need to be addressed. We will use `OneHotEncoder` for conversion and handle missing values based on the `uses_ai` column.

In [30]:
# Cell '3b3506d2' was a duplicate of the preprocessing cell and has been removed for clarity.
# The primary preprocessing is handled by cell '499ca961'.

## 6. Train Machine Learning Models

We train three different regression algorithms: Linear Regression, Random Forest Regressor, and Gradient Boosting Regressor. Each model is fitted to the training data.

In [24]:
# Initialize and train Linear Regression model
print("Training Linear Regression Model...")
linear_reg = LinearRegression()
linear_reg.fit(X_train_processed, y_train)
print("Linear Regression Model trained.")

# Initialize and train Random Forest Regressor model
print("\nTraining Random Forest Regressor Model...")
random_forest_reg = RandomForestRegressor(n_estimators=200, random_state=42)
random_forest_reg.fit(X_train_processed, y_train)
print("Random Forest Regressor Model trained.")

# Initialize and train Gradient Boosting Regressor model
print("\nTraining Gradient Boosting Regressor Model...")
gradient_boost_reg = GradientBoostingRegressor(random_state=42)
gradient_boost_reg.fit(X_train_processed, y_train)
print("Gradient Boosting Regressor Model trained.")

models = {
    "Linear Regression": linear_reg,
    "Random Forest Regressor": random_forest_reg,
    "Gradient Boosting Regressor": gradient_boost_reg
}

Training Linear Regression Model...


NameError: name 'X_train_processed' is not defined

## 7. Make Predictions

After training, we use each model to make predictions on the unseen test dataset. These predictions will be used to evaluate the models' performance.

In [27]:
predictions = {}
for name, model in models.items():
    predictions[name] = model.predict(X_test_processed)
    print(f"Predictions made for {name}")

NameError: name 'models' is not defined

## 8. Evaluate Models

We evaluate each model using several common regression metrics: R² Score, Mean Absolute Error (MAE), Mean Squared Error (MSE), and Root Mean Squared Error (RMSE). These metrics provide a comprehensive understanding of each model's accuracy and error.

In [17]:
evaluation_results = []

for name, y_pred in predictions.items():
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    evaluation_results.append({
        'Model': name,
        'R² Score': r2,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse
    })

    print(f"\n--- {name} --- ")
    print(f"R² Score: {r2:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")

## 9. Compare Models

To facilitate comparison, we consolidate all evaluation metrics into a pandas DataFrame. This table allows us to easily rank models based on their performance, prioritizing higher R² scores and lower RMSE values.

In [18]:
comparison_df = pd.DataFrame(evaluation_results)

# Sort by R² Score (descending) and then by RMSE (ascending)
comparison_df = comparison_df.sort_values(by=['R² Score', 'RMSE'], ascending=[False, True]).reset_index(drop=True)

print("\nModel Comparison Table:")
display(comparison_df)

# Highlight the best performing model (first row after sorting)
best_model_name = comparison_df.iloc[0]['Model']
print(f"\nThe best performing model based on R² Score and RMSE is: {best_model_name}")

KeyError: 'R² Score'

## 10. Visualize Model Performance

Visualizations help to quickly grasp the relative performance of each model. We create bar charts for R² Score, MAE, and RMSE.

In [19]:
plt.style.use('seaborn-v0_8-darkgrid')

# Bar chart for R² Scores
plt.figure(figsize=(12, 6))
sns.barplot(x='Model', y='R² Score', data=comparison_df, palette='viridis')
plt.title('R² Scores of Regression Models', fontsize=16)
plt.xlabel('Model', fontsize=12)
plt.ylabel('R² Score', fontsize=12)
plt.ylim(0, 1) # R² score typically ranges from 0 to 1
for index, row in comparison_df.iterrows():
    plt.text(index, row['R² Score'], f"{row['R² Score']:.3f}", color='black', ha="center", va='bottom', fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Bar chart for MAE
plt.figure(figsize=(12, 6))
sns.barplot(x='Model', y='MAE', data=comparison_df, palette='plasma')
plt.title('Mean Absolute Error (MAE) of Regression Models', fontsize=16)
plt.xlabel('Model', fontsize=12)
plt.ylabel('MAE', fontsize=12)
for index, row in comparison_df.iterrows():
    plt.text(index, row['MAE'], f"{row['MAE']:.3f}", color='black', ha="center", va='bottom', fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Bar chart for RMSE
plt.figure(figsize=(12, 6))
sns.barplot(x='Model', y='RMSE', data=comparison_df, palette='magma')
plt.title('Root Mean Squared Error (RMSE) of Regression Models', fontsize=16)
plt.xlabel('Model', fontsize=12)
plt.ylabel('RMSE', fontsize=12)
for index, row in comparison_df.iterrows():
    plt.text(index, row['RMSE'], f"{row['RMSE']:.3f}", color='black', ha="center", va='bottom', fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

ValueError: Could not interpret value `Model` for `x`. An entry with this name does not appear in `data`.

<Figure size 1200x600 with 0 Axes>

## 11. Select the Best Model

Based on the evaluation metrics, we automatically select the model that achieved the highest R² score. This model is considered the best performer for our prediction task.

In [20]:
best_model_row = comparison_df.iloc[0]
best_model_name = best_model_row['Model']
best_model = models[best_model_name]

print(f"Selected Best Model: {best_model_name}")
print("Performance Metrics of the Best Model:")
print(f"  R² Score: {best_model_row['R² Score']:.4f}")
print(f"  MAE: {best_model_row['MAE']:.4f}")
print(f"  MSE: {best_model_row['MSE']:.4f}")
print(f"  RMSE: {best_model_row['RMSE']:.4f}")
print("\nReason for selection: This model achieved the highest R² score, indicating the best fit to the data, and a relatively low RMSE, suggesting accurate predictions.")

IndexError: single positional indexer is out-of-bounds

## 12. Save the Best Model

The best-performing model is saved using `joblib` to a dedicated `models/` folder. This allows for easy loading and deployment of the trained model without retraining.

In [21]:
import os

# Create a directory for saving models if it doesn't exist
MODEL_DIR = 'models/'
os.makedirs(MODEL_DIR, exist_ok=True)

# Define the path for saving the best model
BEST_MODEL_PATH = os.path.join(MODEL_DIR, 'best_model.pkl')

# Save the best model using joblib
joblib.dump(best_model, BEST_MODEL_PATH)

print(f"Best model '{best_model_name}' saved to {BEST_MODEL_PATH}")

NameError: name 'best_model' is not defined

## 13. Save Additional Objects

If any preprocessing steps involved objects like `StandardScaler` or `OneHotEncoder`, they should also be saved. This ensures consistency when new data is fed to the deployed model. In this specific notebook, no explicit scalers or encoders were used, so this section is conceptual.

```python
# Example of how you would save a scaler if it existed:
# if 'scaler' in locals() and scaler is not None:
#     joblib.dump(scaler, os.path.join(MODEL_DIR, 'scaler.pkl'))
#     print(f"Scaler saved to {os.path.join(MODEL_DIR, 'scaler.pkl')}")
# else:
#     print("No scaler object found to save.")

# Example of how you would save an encoder if it existed:
# if 'encoder' in locals() and encoder is not None:
#     joblib.dump(encoder, os.path.join(MODEL_DIR, 'encoder.pkl'))
#     print(f"Encoder saved to {os.path.join(MODEL_DIR, 'encoder.pkl')}")
# else:
#     print("No encoder object found to save.")
```

*Note: No `scaler` or `encoder` objects were created in this notebook as the data was assumed to be preprocessed already.*

## 14. Test the Saved Model

To ensure the saved model can be loaded correctly and makes consistent predictions, we load it back into memory and use it to predict a few samples from the test set.

In [28]:
# Load the saved model and preprocessor
loaded_model = joblib.load(BEST_MODEL_PATH)
loaded_preprocessor = joblib.load(PREPROCESSOR_PATH)

print(f"Model loaded successfully from {BEST_MODEL_PATH}")
print(f"Preprocessor loaded successfully from {PREPROCESSOR_PATH}")

# Predict a few rows from X_test
sample_X_test = X_test.head(5)

# Preprocess the sample_X_test using the loaded preprocessor and custom imputer
sample_X_test_imputed = custom_imputer(sample_X_test) # Apply the same imputation logic
sample_X_test_processed = loaded_preprocessor.transform(sample_X_test_imputed)

# Convert processed sample back to DataFrame for display (optional)
# 'processed_feature_names' is defined in the preprocessing step
sample_X_test_processed_df = pd.DataFrame(sample_X_test_processed, columns=processed_feature_names, index=sample_X_test.index)

loaded_model_predictions = loaded_model.predict(sample_X_test_processed)

print("\nOriginal X_test samples:")
display(sample_X_test)
print("\nActual y_test values for these samples:")
display(y_test.head(5))
print("\nPredictions from loaded model (on preprocessed samples):")
display(pd.Series(loaded_model_predictions, index=sample_X_test.index, name='Predicted_Grades_After_AI'))

# Verify that the loaded model works by comparing with original model's predictions (optional)
# For a more rigorous check, compare the loaded model's predictions on X_test_processed with `predictions[best_model_name]`
# assert np.array_equal(loaded_model.predict(X_test_processed), predictions[best_model_name])
# print("\nVerification successful: Loaded model produces identical predictions.")

FileNotFoundError: [Errno 2] No such file or directory: 'models/best_model.pkl'

## 15. Conclusion

In this notebook, we successfully trained and evaluated three regression models: Linear Regression, Random Forest Regressor, and Gradient Boosting Regressor, to predict student exam scores. We compared their performance using R² Score, MAE, MSE, and RMSE. The **Random Forest Regressor** model consistently demonstrated the best performance with the highest R² score, making it the selected model for this project. This model was then saved using `joblib` for future use. The next crucial step in this end-to-end ML project will be to deploy this best model using FastAPI, creating a robust and scalable prediction service.